MOSTLY FROM GITHUB: https://github.com/rdbraatz/data-driven-prediction-of-battery-cycle-life-before-capacity-degradation

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle

In [2]:
batch1 = pickle.load(open('../data/batch1.pkl', 'rb'))
#remove batteries that do not reach 80% capacity
del batch1['b1c8']
del batch1['b1c10']
del batch1['b1c12']
del batch1['b1c13']
del batch1['b1c22']

In [3]:
numBat1 = len(batch1.keys())
numBat1

41

In [4]:
batch2 = pickle.load(open('../data/batch2.pkl', 'rb'))

In [5]:
# There are four cells from batch1 that carried into batch2, we'll remove the data from batch2
# and put it with the correct cell from batch1
batch2_keys = ['b2c7', 'b2c8', 'b2c9', 'b2c15', 'b2c16']
batch1_keys = ['b1c0', 'b1c1', 'b1c2', 'b1c3', 'b1c4']
add_len = [662, 981, 1060, 208, 482];

In [6]:
for i, bk in enumerate(batch1_keys):
    batch1[bk]['cycle_life'] = batch1[bk]['cycle_life'] + add_len[i]
    for j in batch1[bk]['summary'].keys():
        if j == 'cycle':
            batch1[bk]['summary'][j] = np.hstack((batch1[bk]['summary'][j], batch2[batch2_keys[i]]['summary'][j] + len(batch1[bk]['summary'][j])))
        else:
            batch1[bk]['summary'][j] = np.hstack((batch1[bk]['summary'][j], batch2[batch2_keys[i]]['summary'][j]))
    last_cycle = len(batch1[bk]['cycles'].keys())
    for j, jk in enumerate(batch2[batch2_keys[i]]['cycles'].keys()):
        batch1[bk]['cycles'][str(last_cycle + j)] = batch2[batch2_keys[i]]['cycles'][jk]

In [7]:
del batch2['b2c7']
del batch2['b2c8']
del batch2['b2c9']
del batch2['b2c15']
del batch2['b2c16']

In [8]:
numBat2 = len(batch2.keys())
numBat2

43

In [9]:
batch3 = pickle.load(open('../data/batch3.pkl', 'rb'))
# remove noisy channels from batch3
del batch3['b3c37']
del batch3['b3c2']
del batch3['b3c23']
del batch3['b3c32']
del batch3['b3c42']
del batch3['b3c43']

In [10]:
numBat3 = len(batch3.keys())
numBat3

40

In [11]:
numBat = numBat1 + numBat2 + numBat3
numBat

124

In [12]:
bat_dict = {**batch1, **batch2, **batch3}

In [13]:
bat_dict.keys()
len(bat_dict.keys())

124

In [14]:
bat_dict["b1c0"].keys()

dict_keys(['cycle_life', 'charge_policy', 'summary', 'cycles'])

In [15]:
def bat_dict_to_df(bat_dict):
    rows = []

    for battery_id, battery_data in bat_dict.items():
        row = {
            "battery_id": battery_id,
            "cycle_life": battery_data["cycle_life"],
            "charge_policy": battery_data["charge_policy"],
            "cycles": battery_data["cycles"],
        }

        # Unravel summary dict into separate columns
        for summary_key, summary_val in battery_data["summary"].items():
            row[summary_key] = np.array(summary_val)

        rows.append(row)

    df = pd.DataFrame(rows)

    return df

In [16]:
bat_df = bat_dict_to_df(bat_dict)

print(bat_df.columns)

Index(['battery_id', 'cycle_life', 'charge_policy', 'cycles', 'IR', 'QC', 'QD',
       'Tavg', 'Tmin', 'Tmax', 'chargetime', 'cycle'],
      dtype='object')


In [17]:
test_ind = np.hstack((np.arange(0,(numBat1+numBat2),2),83))
train_ind = np.arange(1,(numBat1+numBat2-1),2)
secondary_test_ind = np.arange(numBat-numBat3,numBat)

# Split into test, train, and secondary test sets using the same indices
test_df = bat_df.iloc[test_ind]
train_df = bat_df.iloc[train_ind]
secondary_test_df = bat_df.iloc[secondary_test_ind]

In [18]:
print(len(test_df))
print(len(train_df))
print(len(secondary_test_df))

43
41
40


In [19]:
bat_df.to_pickle("../data/bat_df.pkl")

In [ ]:
import os

pkl_path = "../data/bat_df.pkl"

size_bytes = os.path.getsize(pkl_path)

print(f"{size_bytes} bytes")
print(f"{size_bytes / 1024:.2f} KB")
print(f"{size_bytes / 1024**2:.2f} MB")
print(f"{size_bytes / 1024**3:.2f} GB")

7230551726 bytes
7061085.67 KB
6895.59 MB
6.73 GB


7230551726 bytes
7061085.67 KB
6895.59 MB
6.73 GB


In [25]:
from pympler import asizeof

for col in bat_df.columns:
    total = 0

    for item in bat_df[col]:
        total += asizeof.asizeof(item)

    print(f"{col}: {total / 1024**2:.2f} MB")

battery_id: 0.01 MB
cycle_life: 0.00 MB
charge_policy: 0.01 MB
cycles: 14031.03 MB
IR: 0.78 MB
QC: 0.78 MB
QD: 0.78 MB
Tavg: 0.78 MB
Tmin: 0.78 MB
Tmax: 0.78 MB
chargetime: 0.78 MB
cycle: 0.78 MB
